# Elasticsearch RAG ingestion with Google embeddings

This notebook:
1. Loads supply-chain test documents.
2. Chunks the documents.
3. Generates embeddings using Google's embedding API.
4. Creates an Elasticsearch `dense_vector` index.
5. Stores chunks **plus metadata**.
6. Tests semantic search and metadata filtering.

Keep API keys in environment variables; do not commit them to Git.


## 1. Install dependencies

In [ ]:
!pip install -q elasticsearch google-genai python-dotenv tqdm


## 2. Configuration

In [ ]:
import os
from pathlib import Path
from dotenv import load_dotenv
from google import genai
from elasticsearch import Elasticsearch, helpers

load_dotenv()

GOOGLE_API_KEY = os.environ["GOOGLE_API_KEY"]
ELASTICSEARCH_URL = os.environ["ELASTICSEARCH_URL"]
ELASTICSEARCH_API_KEY = os.getenv("ELASTICSEARCH_API_KEY")

ELASTICSEARCH_USERNAME = os.getenv("ELASTICSEARCH_USERNAME")
ELASTICSEARCH_PASSWORD = os.getenv("ELASTICSEARCH_PASSWORD")

# Keep the model configurable because model availability can change.
EMBEDDING_MODEL = os.getenv("EMBEDDING_MODEL", "gemini-embedding-001")
INDEX_NAME = os.getenv("ELASTICSEARCH_INDEX", "supply_chain_rag")

google_client = genai.Client(api_key=GOOGLE_API_KEY)

if ELASTICSEARCH_API_KEY:
    es = Elasticsearch(
        ELASTICSEARCH_URL,
        api_key=ELASTICSEARCH_API_KEY,
    )
else:
    es = Elasticsearch(
        ELASTICSEARCH_URL,
        basic_auth=(ELASTICSEARCH_USERNAME, ELASTICSEARCH_PASSWORD),
    )

print("Elasticsearch connected:", es.ping())
print("Embedding model:", EMBEDDING_MODEL)


## 3. Load test documents

In [ ]:
import json

DATA_DIR = Path("rag_test_data")

documents = [
    json.loads(path.read_text(encoding="utf-8"))
    for path in sorted(DATA_DIR.glob("*.json"))
]

print(f"Loaded {len(documents)} documents")
print(documents[0])


## 4. Chunk documents

In [ ]:
def chunk_text(text: str, chunk_size: int = 80, overlap: int = 15):
    words = text.split()
    chunks = []
    start = 0

    while start < len(words):
        end = min(start + chunk_size, len(words))
        chunks.append(" ".join(words[start:end]))

        if end == len(words):
            break

        start = end - overlap

    return chunks


chunked_documents = []

for doc in documents:
    chunks = chunk_text(doc["content"])

    for chunk_number, chunk in enumerate(chunks):
        chunked_documents.append({
            **{k: v for k, v in doc.items() if k != "content"},
            "chunk_id": f"{doc['document_id']}_chunk_{chunk_number}",
            "chunk_number": chunk_number,
            "content": chunk,
        })

print(f"Created {len(chunked_documents)} chunks")


## 5. Generate Google embeddings

In [ ]:
def embed_text(text: str):
    response = google_client.models.embed_content(
        model=EMBEDDING_MODEL,
        contents=text,
    )
    return response.embeddings[0].values


for item in chunked_documents:
    item["embedding"] = embed_text(item["content"])

embedding_dimension = len(chunked_documents[0]["embedding"])

print("Embedding dimension:", embedding_dimension)


## 6. Create Elasticsearch index

In [ ]:
# For this test notebook we recreate the index.
# In production, use versioned migrations and do not blindly delete indexes.

if es.indices.exists(index=INDEX_NAME):
    es.indices.delete(index=INDEX_NAME)

index_mapping = {
    "settings": {
        "number_of_shards": 1,
        "number_of_replicas": 0,
    },
    "mappings": {
        "properties": {
            "document_id": {"type": "keyword"},
            "chunk_id": {"type": "keyword"},
            "chunk_number": {"type": "integer"},
            "title": {"type": "text"},
            "content": {"type": "text"},
            "document_type": {"type": "keyword"},
            "supplier": {"type": "keyword"},
            "category": {"type": "keyword"},
            "version": {"type": "keyword"},
            "effective_date": {"type": "date"},
            "embedding": {
                "type": "dense_vector",
                "dims": embedding_dimension,
                "index": True,
                "similarity": "cosine",
            },
        }
    },
}

es.indices.create(index=INDEX_NAME, **index_mapping)

print(f"Created index: {INDEX_NAME}")


## 7. Bulk index chunks and metadata

In [ ]:
def actions(items):
    for item in items:
        yield {
            "_index": INDEX_NAME,
            "_id": item["chunk_id"],
            "_source": item,
        }


success, errors = helpers.bulk(
    es,
    actions(chunked_documents),
    raise_on_error=False,
)

print("Indexed:", success)
print("Errors:", len(errors))

if errors:
    print(errors[:2])


## 8. Semantic search

In [ ]:
def semantic_search(query: str, k: int = 5):
    query_vector = embed_text(query)

    response = es.search(
        index=INDEX_NAME,
        knn={
            "field": "embedding",
            "query_vector": query_vector,
            "k": k,
            "num_candidates": max(50, k * 10),
        },
        source=[
            "document_id",
            "chunk_id",
            "title",
            "content",
            "document_type",
            "supplier",
            "category",
            "version",
            "effective_date",
        ],
    )

    return response["hits"]["hits"]


query = "What should happen if a supplier shipment is delayed?"
results = semantic_search(query, k=5)

for hit in results:
    print(f"Score: {hit['_score']:.4f}")
    print(hit["_source"])
    print("-" * 80)


## 9. Semantic search with metadata filtering

In [ ]:
def semantic_search_with_filter(
    query: str,
    k: int = 5,
    supplier: str | None = None,
):
    query_vector = embed_text(query)

    filters = []

    if supplier:
        filters.append({
            "term": {
                "supplier": supplier
            }
        })

    knn = {
        "field": "embedding",
        "query_vector": query_vector,
        "k": k,
        "num_candidates": max(50, k * 10),
    }

    if filters:
        knn["filter"] = filters

    response = es.search(
        index=INDEX_NAME,
        knn=knn,
        source=[
            "document_id",
            "chunk_id",
            "title",
            "content",
            "supplier",
            "category",
        ],
    )

    return response["hits"]["hits"]


results = semantic_search_with_filter(
    "What is the delivery window and delay procedure?",
    k=5,
    supplier="Alpha Components",
)

for hit in results:
    print(hit["_score"], hit["_source"])


## 10. Verify the index

In [ ]:
print("Document count:")
print(es.count(index=INDEX_NAME))

print("\nIndex mapping:")
print(es.indices.get_mapping(index=INDEX_NAME))
